管道提示词可以将多个提示组合在一起。当您想要重复使用部分提示时，这会很有用。这可以通过 PipelinePrompt 来完成。

PipelinePrompt 由两个主要部分组成：
- 最终提示：返回的最终提示
- 管道提示：元组列表，由字符串名称和提示模板组成。每个提示模板将被格式化，然后作为具有相同名称的变量传递到未来的提示模板。

In [1]:
from langchain.prompts.pipeline import PipelinePromptTemplate
from langchain.prompts.prompt import PromptTemplate

In [2]:
full_template = """{introduction}

{example}

{start}"""
full_prompt = PromptTemplate.from_template(full_template)

In [3]:
introduction_template = """你正在冒充{person}。"""
introduction_prompt = PromptTemplate.from_template(introduction_template)

In [4]:
example_template = """
下面是一个交互示例：

Q：{example_q}
A：{example_a}"""
example_prompt = PromptTemplate.from_template(example_template)

In [5]:
start_template = """现在正式开始！

Q：{input}
A："""
start_prompt = PromptTemplate.from_template(start_template)

In [6]:
input_prompts = [
    ("introduction", introduction_prompt),
    ("example", example_prompt),
    ("start", start_prompt),
]
pipeline_prompt = PipelinePromptTemplate(
    final_prompt=full_prompt, pipeline_prompts=input_prompts
)

/tmp/ipykernel_14947/1772176540.py:6: LangChainDeprecationWarning: This class is deprecated in favor of chaining individual prompts together.
  pipeline_prompt = PipelinePromptTemplate(


In [7]:
pipeline_prompt.input_variables

['example_a', 'person', 'example_q', 'input']

In [8]:
print(
    pipeline_prompt.format(
        person="Elon Musk",
        example_q="你最喜欢什么车？",
        example_a="Tesla",
        input="您最喜欢的社交媒体网站是什么?",
    )
)

你正在冒充Elon Musk。


下面是一个交互示例：

Q：你最喜欢什么车？
A：Tesla

现在正式开始！

Q：您最喜欢的社交媒体网站是什么?
A：


In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:1234/v1"
chat = ChatOpenAI(
    openai_api_key=openai_api_key,
    openai_api_base=openai_api_base,
    temperature=0.7,
)

output_parser = StrOutputParser()

chain = pipeline_prompt | chat | output_parser

chain.invoke({
    "input":"您最喜欢的社交媒体网站是什么",
    "person":"Elon Musk",
    "example_q":"你最喜欢什么车？",
    "example_a":"Tesla",
})

'<think>\n好的，用户让我扮演的是Elon Musk，现在需要回答他最喜欢的社交媒体网站。首先，我得考虑Elon Musk作为企业家和科技巨头的背景，他的偏好可能偏向于科技相关的平台。\n\n作为一个以推动电动汽车、太空探索和可持续发展著称的领导者，Elon Musk可能会关注那些能够促进创新、环保和社会变革的社交媒体平台。考虑到他的公司如Tesla、SpaceX以及Boring Company等，他可能更倾向于使用那些能促进技术交流和合作的平台。\n\n在这样的背景下，最有可能的社交媒体网站是Twitter（x），因为这是Elon Musk所运营的主要平台，用于发布公司动态、推动项目进展，并与公众互动。此外，考虑到他的多个项目，如Neuralink、The Boring Company等，他可能也会在其他平台上进行活动，但Twitter作为他主要的沟通渠道，应该是最直接的答案。\n\n不过，我需要确认的是，虽然Elon Musk确实使用Twitter，但是否还有其他的社交媒体平台更符合他的偏好。比如，LinkedIn也是一个专业的社交网络，但通常用于职业发展和行业交流。对于一个科技企业家来说，LinkedIn可能也是重要的，但主要的沟通渠道还是Twitter。\n\n因此，最合适的回答应该是Twitter（x）。不过，也可能有其他可能性，比如Facebook或Instagram，但这不太可能。因为Elon Musk更倾向于使用技术驱动的平台，而Instagram更多用于个人形象和品牌推广，虽然他本人也经常在Instagram上活动，但主要的沟通渠道还是Twitter。\n\n所以最终答案应该是Twitter（x）。\n</think>\n\nA：Twitter（x）'